# Deep Learning with PyTorch

**Module 03: Deep Learning**  
**Objective**: Modern deep learning with PyTorch, batch norm, dropout, and advanced techniques

## What You'll Learn
1. Building modern architectures with PyTorch
2. Batch Normalization
3. Dropout and regularization
4. Learning rate scheduling
5. Data augmentation
6. Transfer learning
7. Model checkpointing and early stopping
8. TensorBoard logging

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Modern MLP with Batch Norm and Dropout

In [ ]:
class ModernMLP(nn.Module):
    """MLP with BatchNorm, Dropout, and Residual Connections."""
    
    def __init__(self, input_dim, hidden_dims, output_dim, dropout=0.3):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, output_dim))
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

# Model
model = ModernMLP(input_dim=784, hidden_dims=[512, 256, 128], output_dim=10, dropout=0.3).to(device)
print(f"\nModel:\n{model}")

# Training
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for X, y in loader:
        X = X.view(-1, 784).to(device)
        y = y.to(device)
        
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()
    
    return total_loss / len(loader), 100. * correct / total

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for X, y in loader:
            X = X.view(-1, 784).to(device)
            y = y.to(device)
            
            outputs = model(X)
            loss = criterion(outputs, y)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()
    
    return total_loss / len(loader), 100. * correct / total

# Train
epochs = 10
train_losses, train_accs = [], []
test_losses, test_accs = [], []

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    test_loss, test_acc = evaluate(model, test_loader, criterion)
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    test_losses.append(test_loss)
    test_accs.append(test_acc)
    
    print(f"Epoch {epoch+1}/{epochs}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.2f}%, "
          f"Test Loss={test_loss:.4f}, Test Acc={test_acc:.2f}%")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, label='Train')
ax1.plot(test_losses, label='Test')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Curves')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(train_accs, label='Train')
ax2.plot(test_accs, label='Test')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy Curves')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Learning Rate Schedulers

In [ ]:
# Compare different schedulers
model = ModernMLP(784, [256, 128], 10).to(device)
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

schedulers = {
    'StepLR': optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1),
    'ExponentialLR': optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9),
    'CosineAnnealingLR': optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10),
    'ReduceLROnPlateau': optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)
}

# Visualize
plt.figure(figsize=(10, 6))

for name, scheduler in schedulers.items():
    lrs = []
    temp_optimizer = optim.SGD(model.parameters(), lr=0.1)
    
    if name == 'StepLR':
        temp_scheduler = optim.lr_scheduler.StepLR(temp_optimizer, step_size=3, gamma=0.1)
    elif name == 'ExponentialLR':
        temp_scheduler = optim.lr_scheduler.ExponentialLR(temp_optimizer, gamma=0.9)
    elif name == 'CosineAnnealingLR':
        temp_scheduler = optim.lr_scheduler.CosineAnnealingLR(temp_optimizer, T_max=10)
    else:
        continue  # ReduceLROnPlateau needs validation loss
    
    for epoch in range(20):
        lrs.append(temp_optimizer.param_groups[0]['lr'])
        temp_scheduler.step()
    
    plt.plot(lrs, label=name, marker='o')

plt.xlabel('Epoch')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedules')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

print("StepLR: Reduces by factor at fixed intervals")
print("ExpLR: Exponential decay")
print("CosineAnnealing: Cosine decay (popular for transformers)")
print("ReduceLROnPlateau: Reduce when validation plateaus")

## 3. Early Stopping and Model Checkpointing

In [ ]:
class EarlyStopping:
    """Early stopping to prevent overfitting."""
    
    def __init__(self, patience=5, min_delta=0, path='checkpoint.pt'):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.path = path
    
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter}/{self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.save_checkpoint(model)
            self.counter = 0
    
    def save_checkpoint(self, model):
        """Save model checkpoint."""
        torch.save(model.state_dict(), self.path)
        print(f'Checkpoint saved (val_loss={self.best_loss:.4f})')

# Example usage
model = ModernMLP(784, [256, 128], 10).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
early_stopping = EarlyStopping(patience=3, path='best_model.pt')

for epoch in range(20):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = evaluate(model, test_loader, criterion)
    
    print(f"Epoch {epoch+1}: Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%")
    
    early_stopping(val_loss, model)
    
    if early_stopping.early_stop:
        print("Early stopping triggered!")
        break

# Load best model
model.load_state_dict(torch.load('best_model.pt'))
final_loss, final_acc = evaluate(model, test_loader, criterion)
print(f"\nBest model - Test Loss: {final_loss:.4f}, Test Acc: {final_acc:.2f}%")

## 4. Data Augmentation

In [ ]:
# Data augmentation for robustness
augmented_transform = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

augmented_dataset = datasets.MNIST('./data', train=True, download=True, transform=augmented_transform)
augmented_loader = DataLoader(augmented_dataset, batch_size=128, shuffle=True)

# Visualize augmentations
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    img, label = train_dataset[i]
    axes[0, i].imshow(img.squeeze(), cmap='gray')
    axes[0, i].set_title(f'Original: {label}')
    axes[0, i].axis('off')
    
    img_aug, _ = augmented_dataset[i]
    axes[1, i].imshow(img_aug.squeeze(), cmap='gray')
    axes[1, i].set_title('Augmented')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

# Train with augmentation
model_aug = ModernMLP(784, [256, 128], 10).to(device)
optimizer_aug = optim.Adam(model_aug.parameters(), lr=0.001)

print("\nTraining with augmented data...")
for epoch in range(5):
    train_loss, train_acc = train_epoch(model_aug, augmented_loader, criterion, optimizer_aug)
    test_loss, test_acc = evaluate(model_aug, test_loader, criterion)
    print(f"Epoch {epoch+1}: Test Acc={test_acc:.2f}%")

## Summary

Modern deep learning techniques:
- ✅ Batch Normalization (stabilizes training)
- ✅ Dropout (prevents overfitting)
- ✅ Learning rate scheduling (improves convergence)
- ✅ Early stopping (prevents overfitting)
- ✅ Model checkpointing (save best model)
- ✅ Data augmentation (improves generalization)

**Next Module**: CNNs for computer vision!